# Chunk

In [23]:
# cell 1

import os
from pathlib import Path

import pandas as pd

# -------------------------------------------------------------------
# PROJECT PATHS
# -------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()

RAW_DIR = PROJECT_ROOT / "00_data" / "raw"  # where arcos_all.tsv lives
PROCESSED_DIR = PROJECT_ROOT / "00_data" / "processed"
DOCS_DIR = PROJECT_ROOT / "00_data" / "docs"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

# Path to the huge ARCOS shipment file you downloaded
ARCOS_FILE = RAW_DIR / "arcos_all.tsv"

# -------------------------------------------------------------------
# COLUMN MAP – UPDATE THESE IF YOUR RAW FILE USES DIFFERENT NAMES
# -------------------------------------------------------------------
COLS = {
    # geography
    "state": "BUYER_STATE",
    "county": "BUYER_COUNTY",  # or county FIPS if that exists
    "zip": "BUYER_ZIP",  # optional
    # timing
    "date": "TRANSACTION_DATE",  # often YYYYMMDD or date string
    # drug info
    "drug_name": "DRUG_NAME",
    "drug_code": "DRUG_CODE",  # optional, not used in our aggregation
    # shipment volume
    "quantity": "DOSAGE_UNIT",  # count of pills / units dispensed
    # MME-related columns
    # NOTE: In the FOIA file we have both:
    #   - CALC_BASE_WT_IN_GM        (grams of active ingredient)
    #   - MME_Conversion_Factor     (MME per mg)
    # We will *prefer* to compute MME from those, instead of trusting the raw "MME" column.
    "mme": "MME",  # still listed here as a fallback candidate
    # Only needed if you must compute MME from pill strength:
    "strength_mg": "Dosage_Strength",  # mg per unit (if available)
    "mme_factor": "MME_Conversion_Factor",  # MME per mg
}

In [24]:
# cell 2

from typing import Optional


def load_data_dictionary(path: Path | str) -> pd.DataFrame:
    """Load the ARCOS data dictionary for reference (optional)."""
    path = Path(path)
    return pd.read_csv(path)


def compute_mme(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure we have an 'mme' column (Morphine Milligram Equivalents, *mg*).

    Priority:
    1. If the raw file has CALC_BASE_WT_IN_GM (grams of base) and an
       MME conversion factor (MME_Conversion_Factor / MME_CONVERSION_FACTOR),
       compute:
           mme (mg) = CALC_BASE_WT_IN_GM * 1000 * factor
       This is the most interpretable and avoids weird scaling in the
       precomputed 'MME' column.
    2. Otherwise, if a precomputed MME column exists (COLS["mme"], CALC_MME, MME),
       rename it to 'mme' and use it directly.
    3. Otherwise, fall back to:
           mme = quantity * strength_mg * mme_factor
       where all units are assumed to be in mg.
    """

    # --------------------------------------------------
    # Case 1: use base-weight in grams + conversion factor
    # --------------------------------------------------
    base_candidates = ["CALC_BASE_WT_IN_GM", "Calc_Base_Wt_in_gm"]
    factor_candidates = ["MME_Conversion_Factor", "MME_CONVERSION_FACTOR"]

    base_col = next((c for c in base_candidates if c in df.columns), None)
    factor_col = next((c for c in factor_candidates if c in df.columns), None)

    if base_col is not None and factor_col is not None:
        base_g = pd.to_numeric(df[base_col], errors="coerce")  # grams of base
        factor = pd.to_numeric(df[factor_col], errors="coerce")  # MME per mg

        # Convert grams -> mg, then apply MME factor
        df["mme"] = base_g * 1000.0 * factor
        return df

    # --------------------------------------------------
    # Case 2: precomputed MME column present
    # --------------------------------------------------
    mme_col_candidates = [COLS.get("mme"), "CALC_MME", "MME"]  # common names
    mme_col_candidates = [c for c in mme_col_candidates if c is not None]

    existing_mme_cols = [c for c in mme_col_candidates if c in df.columns]

    if existing_mme_cols:
        mme_col = existing_mme_cols[0]
        df = df.rename(columns={mme_col: "mme"})
        df["mme"] = pd.to_numeric(df["mme"], errors="coerce")
        return df

    # --------------------------------------------------
    # Case 3: compute from quantity * strength * factor
    # --------------------------------------------------
    qty_col = COLS.get("quantity")
    strength_col = COLS.get("strength_mg")
    factor_col = COLS.get("mme_factor")

    missing = [c for c in [qty_col, strength_col, factor_col] if c not in df.columns]
    if missing:
        raise ValueError(
            f"Cannot compute MME: missing columns {missing}. "
            "Either provide a precomputed MME column or update COLS mapping."
        )

    df["mme"] = (
        pd.to_numeric(df[qty_col], errors="coerce")
        * pd.to_numeric(df[strength_col], errors="coerce")
        * pd.to_numeric(df[factor_col], errors="coerce")
    )

    return df


def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Rename key columns to simple names used downstream.

    We only rename if the raw columns are present to avoid KeyErrors.
    """
    rename_map: dict[str, str] = {}

    if COLS["state"] in df.columns:
        rename_map[COLS["state"]] = "state"
    if COLS["county"] in df.columns:
        rename_map[COLS["county"]] = "county"
    if COLS["date"] in df.columns:
        rename_map[COLS["date"]] = "date"
    if COLS["drug_name"] in df.columns:
        rename_map[COLS["drug_name"]] = "drug_name"
    if COLS["drug_code"] in df.columns:
        rename_map[COLS["drug_code"]] = "drug_code"
    if COLS["quantity"] in df.columns:
        rename_map[COLS["quantity"]] = "qty_pills"

    df = df.rename(columns=rename_map)
    return df

In [25]:
# cell 3

from typing import Iterable


def arcos_chunks(
    path: Path | str, chunksize: int = 1_000_000
) -> Iterable[pd.DataFrame]:
    """
    Generator that yields chunks of the ARCOS TSV file.

    We read with sep='\\t' because the FOIA file is tab-separated.
    """
    return pd.read_csv(
        path,
        sep="\t",
        chunksize=chunksize,
        low_memory=False,
    )

In [26]:
# cell 4

from tqdm.auto import tqdm


def process_to_county_year(path: Path) -> pd.DataFrame:
    """
    Read the ARCOS file in chunks and return a county-year dataset with:
      - total_grams     : total grams of active ingredient
      - total_pills     : total number of dosage units
      - total_mme       : total Morphine Milligram Equivalents (mg)
      - n_transactions  : number of shipment records

    IMPORTANT:
    ----------
    We do *NOT* filter to specific molecule names (e.g. oxycodone/hydrocodone).
    Per professor guidance, the vast majority of ARCOS shipments in this period
    are prescription opioids, and we use total MME to capture potency-adjusted
    opioid supply.

    Aggregation level:
      (state, county, year)
    """

    aggregates: list[pd.DataFrame] = []

    for chunk in tqdm(arcos_chunks(path), desc="Processing ARCOS"):
        # --- 1. Standardize column names ---
        chunk = standardize_columns(chunk)

        # Sanity check: ensure required cols exist
        required = {"state", "county", "date", "qty_pills"}
        missing = required - set(chunk.columns)
        if missing:
            raise ValueError(f"Missing required columns in chunk: {missing}")

        # --- 2. Clean text fields ---
        chunk["state"] = chunk["state"].astype(str).str.upper().str.strip()
        chunk["county"] = chunk["county"].astype(str).str.upper().str.strip()

        if "drug_name" in chunk.columns:
            chunk["drug_name"] = chunk["drug_name"].astype(str).str.upper().str.strip()
        else:
            # If drug_name not present, create a dummy for counting transactions
            chunk["drug_name"] = "UNKNOWN"

        # --- 3. Parse date → year ---
        chunk["date"] = pd.to_datetime(chunk["date"], errors="coerce")
        chunk["year"] = chunk["date"].dt.year

        # Drop rows with missing key fields
        chunk = chunk.dropna(subset=["state", "county", "year"])

        # --- 4. Ensure numeric quantity + MME ---
        chunk["qty_pills"] = pd.to_numeric(chunk["qty_pills"], errors="coerce")

        # Compute or normalize MME (mg)
        chunk = compute_mme(chunk)
        chunk["mme"] = pd.to_numeric(chunk["mme"], errors="coerce")

        # Also try to keep a "grams of base" column if available,
        # so we can aggregate total grams at the county-year level.
        base_candidates = ["CALC_BASE_WT_IN_GM", "Calc_Base_Wt_in_gm"]
        base_col = next((c for c in base_candidates if c in chunk.columns), None)
        if base_col is not None:
            chunk["grams"] = pd.to_numeric(chunk[base_col], errors="coerce")
        else:
            # If not available, we can back out grams from MME and factor
            # but for now we just set grams = NaN and only MME will be used.
            chunk["grams"] = pd.NA

        # Drop rows with no quantity or MME
        chunk = chunk.dropna(subset=["qty_pills", "mme"])

        if chunk.empty:
            continue

        # --- 5. Group to county-year for this chunk ---
        grouped = chunk.groupby(["state", "county", "year"], as_index=False).agg(
            total_grams=("grams", "sum"),
            total_pills=("qty_pills", "sum"),
            total_mme=("mme", "sum"),
            n_transactions=("drug_name", "size"),
        )
        aggregates.append(grouped)

    if not aggregates:
        raise ValueError("No data processed – check column mappings or file path.")

    # Combine across all chunks
    county_year = pd.concat(aggregates, ignore_index=True)

    # Ensure uniqueness across chunks (sum repeated keys)
    county_year = county_year.groupby(["state", "county", "year"], as_index=False).agg(
        total_grams=("total_grams", "sum"),
        total_pills=("total_pills", "sum"),
        total_mme=("total_mme", "sum"),
        n_transactions=("n_transactions", "sum"),
    )

    return county_year

/Users/sungtsewu/miniforge3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [27]:
# cell 5

county_year = process_to_county_year(ARCOS_FILE)

print("Final county-year shape:", county_year.shape)
display(county_year.head())

# Save in both CSV and Parquet
out_csv = PROCESSED_DIR / "new_arcos_county_year.csv"
out_parquet = PROCESSED_DIR / "new_arcos_county_year.parquet"

county_year.to_csv(out_csv, index=False)
county_year.to_parquet(out_parquet, index=False)

print("Saved:")
print(" -", out_csv)
print(" -", out_parquet)

Processing ARCOS: 760it [1:33:40,  7.40s/it]


Final county-year shape: (43822, 7)


,state,county,year,total_grams,total_pills,total_mme,n_transactions
0,AA,NAN,2009,0.060540,10.0,60.540,2
1,AA,NAN,2010,0.036324,6.0,36.324,1
2,AE,NAN,2006,1.362150,330.0,1362.150,2
3,AK,ALEUTIANS EAST,2006,0.908100,300.0,908.100,1
4,AK,ALEUTIANS EAST,2007,0.302700,100.0,302.700,1


Saved:
 - /Volumes/Jay_SSD/studying/duke-mids-courses/IDS720/Opioids/00_data/processed/new_arcos_county_year.csv
 - /Volumes/Jay_SSD/studying/duke-mids-courses/IDS720/Opioids/00_data/processed/new_arcos_county_year.parquet


In [28]:
df = pd.read_csv(out_csv)

target_states = ["FL", "GA", "AL", "SC", "WA", "OR", "ID", "CO"]

df_subset = df[df["state"].isin(target_states)]

df_subset.shape, df_subset.head()

((7124, 7),
     state   county  year   total_grams  total_pills     total_mme  \
 330    AL  AUTAUGA  2006  24509.969655  2850573.246  5.166274e+07   
 331    AL  AUTAUGA  2007  25952.872621  3167354.666  5.309262e+07   
 332    AL  AUTAUGA  2008  28680.180783  3453070.648  5.746818e+07   
 333    AL  AUTAUGA  2009  31008.028184  3759035.404  5.830612e+07   
 334    AL  AUTAUGA  2010  34013.266155  3841098.472  6.263222e+07   
 
      n_transactions  
 330            8838  
 331            9760  
 332           10389  
 333           10528  
 334           11549  )

In [ ]:
states_out_csv = PROCESSED_DIR / "new_arcos_county_year_8states.csv"

df_subset.to_csv(states_out_csv, index=False)

=========================================================================================================

# Merge

In [30]:
# CELL 1: imports and paths

import pandas as pd
import numpy as np
from pathlib import Path

# Assume this notebook is in `notebooks/` – go up one level to repo root
PROJECT_ROOT = Path("..").resolve()

DATA_CLEAN = PROJECT_ROOT / "00_data" / "clean"
DATA_PROCESSED = PROJECT_ROOT / "00_data" / "processed"

# States in our analysis
KEEP_STATES = ["FL", "GA", "AL", "SC", "WA", "CO", "ID", "OR"]

In [31]:
# CELL 2: helper functions


def std_state_abbrev(s: pd.Series) -> pd.Series:
    """Standardize state abbreviations (upper + strip)."""
    return s.astype(str).str.upper().str.strip()


def std_county(c: pd.Series) -> pd.Series:
    """Standardize county names so they match across datasets."""
    return (
        c.astype(str)
        .str.upper()
        .str.strip()
        .str.replace(" COUNTY", "", regex=False)
        .str.replace(" PARISH", "", regex=False)
        .str.replace(" BOROUGH", "", regex=False)
        .str.replace(" CENSUS AREA", "", regex=False)
    )

In [ ]:
# CELL 3: load county population (wide -> long)

pop_path = DATA_CLEAN / "county_population_2000_2015.csv"
pop_wide = pd.read_csv(pop_path)

print("Population columns:", list(pop_wide.columns))
print("Population shape BEFORE reshape:", pop_wide.shape)

year_cols = [c for c in pop_wide.columns if c.startswith("POPESTIMATE")]
print("Year columns detected:", year_cols)
if not year_cols:
    raise ValueError("No POPESTIMATE* columns found – check the population file.")

id_cols = ["STATE_ABBR", "CTYNAME"]
keep_cols = id_cols + year_cols
pop_wide = pop_wide[keep_cols].copy()

pop_long = pop_wide.melt(
    id_vars=id_cols,
    value_vars=year_cols,
    var_name="year_col",
    value_name="population",
)

pop_long["year"] = pop_long["year_col"].str.extract(r"(\d{4})").astype(int)
pop_long = pop_long.drop(columns=["year_col"])

pop = pop_long.rename(
    columns={
        "STATE_ABBR": "state",
        "CTYNAME": "county",
    }
)

pop["state"] = std_state_abbrev(pop["state"])
pop["county"] = std_county(pop["county"])

pop = pop[(pop["year"] >= 2006) & (pop["year"] <= 2015)]
pop = pop[pop["state"].isin(KEEP_STATES)].copy()

print("Population shape AFTER reshape + filter:", pop.shape)
pop.head()

Population columns: ['SUMLEV', 'REGION', 'DIVISION', 'STATE', 'COUNTY', 'STNAME', 'CTYNAME', 'ESTIMATESBASE2000', 'POPESTIMATE2001', 'POPESTIMATE2002', 'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005', 'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008', 'POPESTIMATE2009', 'CENSUS2010POP', 'POPESTIMATE2010', 'STATE_FIPS', 'COUNTY_FIPS', 'FIPS', 'POPESTIMATE2011', 'POPESTIMATE2012', 'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015', 'STATE_ABBR']
Population shape BEFORE reshape: (3112, 28)
Year columns detected: ['POPESTIMATE2001', 'POPESTIMATE2002', 'POPESTIMATE2003', 'POPESTIMATE2004', 'POPESTIMATE2005', 'POPESTIMATE2006', 'POPESTIMATE2007', 'POPESTIMATE2008', 'POPESTIMATE2009', 'POPESTIMATE2010', 'POPESTIMATE2011', 'POPESTIMATE2012', 'POPESTIMATE2013', 'POPESTIMATE2014', 'POPESTIMATE2015']
Population shape AFTER reshape + filter: (5220, 4)


,state,county,population,year
15560,AL,AUTAUGA,51328,2006
15561,AL,BALDWIN,168121,2006
15562,AL,BARBOUR,27861,2006
15563,AL,BIBB,22099,2006
15564,AL,BLOUNT,55485,2006


In [ ]:
# CELL 4: load county mortality

mort_path = DATA_CLEAN / "df_overdose_cleaned.csv"
mort = pd.read_csv(mort_path)

print("Mortality columns:", mort.columns)
print("Mortality shape BEFORE cleaning:", mort.shape)

rename_map_mort = {
    "state": "state",
    "county": "county",
    "year": "year",
    "deaths": "drug_deaths",
}

mort = mort.rename(columns=rename_map_mort)

mort["state"] = std_state_abbrev(mort["state"])
mort["county"] = std_county(mort["county"])
mort["year"] = mort["year"].astype(int)

mort = mort[(mort["year"] >= 2006) & (mort["year"] <= 2015)]
mort = mort[mort["state"].isin(KEEP_STATES)].copy()

print("Mortality shape AFTER filter:", mort.shape)
mort.head()

Mortality columns: Index(['county', 'county_code', 'year', 'drug/alcohol_induced_cause',
       'drug/alcohol_induced_cause_code', 'deaths', 'state'],
      dtype='object')
Mortality shape BEFORE cleaning: (6300, 7)
Mortality shape AFTER filter: (1281, 7)


,county,county_code,year,drug/alcohol_induced_cause,drug/alcohol_induced_cause_code,drug_deaths,state
0,BALDWIN,1003,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,29.0,AL
1,CALHOUN,1015,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,12.0,AL
2,ETOWAH,1055,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,14.0,AL
3,HOUSTON,1069,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,10.0,AL
4,JEFFERSON,1073,2009,Drug poisonings (overdose) Unintentional (X40-...,D1,80.0,AL


In [ ]:
# CELL 5: load ARCOS county-year summary

arcos_path = DATA_PROCESSED / "new_arcos_county_year_8states.csv"
arcos = pd.read_csv(arcos_path)

print("ARCOS columns:", arcos.columns)
print("ARCOS shape BEFORE cleaning:", arcos.shape)

arcos["state"] = std_state_abbrev(arcos["state"])
arcos["county"] = std_county(arcos["county"])
arcos["year"] = arcos["year"].astype(int)

arcos = arcos[arcos["state"].isin(KEEP_STATES)].copy()
arcos = arcos[(arcos["year"] >= 2006) & (arcos["year"] <= 2015)]

print("ARCOS shape AFTER filter:", arcos.shape)
arcos.head()

ARCOS columns: Index(['state', 'county', 'year', 'total_grams', 'total_pills', 'total_mme',
       'n_transactions'],
      dtype='object')
ARCOS shape BEFORE cleaning: (7124, 7)
ARCOS shape AFTER filter: (5086, 7)


,state,county,year,total_grams,total_pills,total_mme,n_transactions
0,AL,AUTAUGA,2006,24509.969655,2850573.246,5.166274e+07,8838
1,AL,AUTAUGA,2007,25952.872621,3167354.666,5.309262e+07,9760
2,AL,AUTAUGA,2008,28680.180783,3453070.648,5.746818e+07,10389
3,AL,AUTAUGA,2009,31008.028184,3759035.404,5.830612e+07,10528
4,AL,AUTAUGA,2010,34013.266155,3841098.472,6.263222e+07,11549


In [ ]:
# CELL 6: merge population, ARCOS, and mortality → county-year panel

panel = pop.copy()

panel = panel.merge(
    arcos[
        [
            "state",
            "county",
            "year",
            "total_grams",
            "total_pills",
            "total_mme",
            "n_transactions",
        ]
    ],
    on=["state", "county", "year"],
    how="left",
    validate="1:1",
)

panel = panel.merge(
    mort[["state", "county", "year", "drug_deaths"]],
    on=["state", "county", "year"],
    how="left",
    validate="1:1",
)

print("Panel shape after merges:", panel.shape)
panel[
    ["state", "county", "year", "population", "total_pills", "total_mme", "drug_deaths"]
].head()

Panel shape after merges: (5220, 9)


,state,county,year,population,total_pills,total_mme,drug_deaths
0,AL,AUTAUGA,2006,51328,2850573.246,5.166274e+07,NaN
1,AL,BALDWIN,2006,168121,8586086.494,1.628179e+08,11.0
2,AL,BARBOUR,2006,27861,1018381.979,1.169558e+07,NaN
3,AL,BIBB,2006,22099,962487.912,1.658575e+07,NaN
4,AL,BLOUNT,2006,55485,1590574.274,2.555605e+07,NaN


In [ ]:
# CELL 7: feature engineering at county-year level

panel["missing_deaths"] = panel["drug_deaths"].isna().astype(int)

# --- per capita metrics (county level) ---
panel["mme_per_capita"] = panel["total_mme"] / panel["population"]
panel["pills_per_capita"] = panel["total_pills"] / panel["population"]
panel["overdoses_per_100k"] = (panel["drug_deaths"] / panel["population"]) * 100_000
panel["log_mme_per_capita"] = np.log1p(panel["mme_per_capita"])
panel["log_overdoses_per_100k"] = np.log1p(panel["overdoses_per_100k"])

panel[["state", "county", "year", "mme_per_capita", "overdoses_per_100k"]].head()

,state,county,year,mme_per_capita,overdoses_per_100k
0,AL,AUTAUGA,2006,1006.521665,NaN
1,AL,BALDWIN,2006,968.456784,6.542907
2,AL,BARBOUR,2006,419.783224,NaN
3,AL,BIBB,2006,750.520260,NaN
4,AL,BLOUNT,2006,460.593891,NaN


In [ ]:
# CELL 8: aggregate to state-year level for plotting (shipment + mortality)

state_year = panel.groupby(["state", "year"], as_index=False).agg(
    total_mme_state=("total_mme", "sum"),
    total_pills_state=("total_pills", "sum"),
    total_pop_state=("population", "sum"),
    total_drug_deaths_state=("drug_deaths", "sum"),
)

state_year["mme_per_capita"] = (
    state_year["total_mme_state"] / state_year["total_pop_state"]
)

state_year["mke_per_1M"] = state_year["mme_per_capita"]

state_year["overdoses_per_100k_state"] = (
    state_year["total_drug_deaths_state"] / state_year["total_pop_state"]
) * 100_000

state_year.head()

,state,year,total_mme_state,total_pills_state,total_pop_state,total_drug_deaths_state,mme_per_capita,mke_per_1M,overdoses_per_100k_state
0,AL,2006,1.934108e+10,2.820718e+09,4628981,165.0,4178.258823,4178.258823,3.564499
1,AL,2007,2.729516e+10,4.435053e+09,4672840,287.0,5841.234854,5841.234854,6.141875
2,AL,2008,2.658357e+10,3.817756e+09,4718206,361.0,5634.254457,5634.254457,7.651213
3,AL,2009,3.930749e+10,4.513204e+09,4757938,389.0,8261.455165,8261.455165,8.175811
4,AL,2010,4.353394e+10,5.630678e+09,4785298,321.0,9097.436017,9097.436017,6.708046


In [38]:
# CELL 9: save master panels

# county-year panel
out_county_csv = DATA_PROCESSED / "new_master_county_year_panel.csv"
out_county_parquet = DATA_PROCESSED / "new_master_county_year_panel.parquet"

panel.to_csv(out_county_csv, index=False)
panel.to_parquet(out_county_parquet, index=False)

# state-year summary (for plotting shipment / mortality)
out_state_csv = DATA_PROCESSED / "new_state_year_shipment_mortality.csv"
out_state_parquet = DATA_PROCESSED / "new_state_year_shipment_mortality.parquet"

state_year.to_csv(out_state_csv, index=False)
state_year.to_parquet(out_state_parquet, index=False)

print("Saved:")
print(" -", out_county_csv)
print(" -", out_county_parquet)
print(" -", out_state_csv)
print(" -", out_state_parquet)

panel.shape, state_year.shape

Saved:
 - /Volumes/Jay_SSD/studying/duke-mids-courses/IDS720/Opioids/00_data/processed/new_master_county_year_panel.csv
 - /Volumes/Jay_SSD/studying/duke-mids-courses/IDS720/Opioids/00_data/processed/new_master_county_year_panel.parquet
 - /Volumes/Jay_SSD/studying/duke-mids-courses/IDS720/Opioids/00_data/processed/new_state_year_shipment_mortality.csv
 - /Volumes/Jay_SSD/studying/duke-mids-courses/IDS720/Opioids/00_data/processed/new_state_year_shipment_mortality.parquet


((5220, 15), (80, 9))